In [ ]:
using Revise

In [2]:
using GeneRegulatorySystems
using GeneRegulatorySystems.Models
using GeneRegulatorySystems.Models.NetworkRepresentation
using GeneRegulatorySystems.Models.V1
using GeneRegulatorySystems.Scheduling
import JSON
using GarishPrint

macro pp(x) :(GarishPrint.pprint($(esc(x)))) end;

In [9]:
# Test ACDC (has proteolysis)
using Catalyst

schedule_acdc = Models.load("examples/toy/ACDC.schedule.json", seed="seed123")

# Get a model path from dryrun
model_path = nothing
schedule_acdc(Models.FlatState(); dryrun = (primitive!, x, dt; path, _...) -> begin
    if isfinite(dt) && dt > 0.0 && model_path === nothing
        global model_path = primitive!.path
    end
end)
println("Model path: $model_path")

wrapped = Scheduling.reify(schedule_acdc, model_path)
v1_def = wrapped.model.definition
rs = wrapped.model.model.definition

all_rxns = Catalyst.reactions(rs)
reg_filtered = [rxn for rxn in all_rxns if NetworkRepresentation._is_regulation_reaction(rxn)]
non_reg = [rxn for rxn in all_rxns if !NetworkRepresentation._is_regulation_reaction(rxn)]

desc = Models.describe(v1_def)
components = Dict(typeof(d) => d for d in desc.descriptions)
raw_links = components[Models.Network].links
gene_names = Set{Symbol}(g.name for g in v1_def.genes)
prot_ids = NetworkRepresentation._proteolysis_reaction_ids(raw_links, gene_names)

prot_filtered = [rxn for rxn in non_reg if NetworkRepresentation._reaction_id(rxn) in prot_ids]
surviving = [rxn for rxn in non_reg if !(NetworkRepresentation._reaction_id(rxn) in prot_ids)]

println("Total reactions: $(length(all_rxns))")
println("Regulation-filtered: $(length(reg_filtered))")
println("Proteolysis IDs computed: $(length(prot_ids))")
println("Proteolysis-filtered: $(length(prot_filtered))")
println("Surviving: $(length(surviving))")

println("\n=== Proteolysis IDs ===")
for id in prot_ids
    println("  $id")
end

println("\n=== Proteolysis-filtered reactions ===")
for rxn in prot_filtered
    subs = [String(NetworkRepresentation.SpeciesId(s).name) for s in rxn.substrates]
    prods = [String(NetworkRepresentation.SpeciesId(p).name) for p in rxn.products]
    rxn_id = NetworkRepresentation._reaction_id(rxn)
    println("  $(join(subs, " + ")) -> $(join(prods, " + "))  [id=$rxn_id]")
end

e = NetworkRepresentation.entity(wrapped)
nodes, links = NetworkRepresentation.flatten(e)
println("\n=== Final: $(count(n->n.kind==:gene, nodes)) genes, $(count(n->n.kind==:species, nodes)) species, $(count(n->n.kind==:reaction, nodes)) reactions ===")

Model path: -2-1/1.do
Total reactions: 30
Regulation-filtered: 6
Proteolysis IDs computed: 0
Proteolysis-filtered: 0
Surviving: 24

=== Proteolysis IDs ===

=== Proteolysis-filtered reactions ===

=== Final: 3 genes, 18 species, 24 reactions ===


Load schedule

In [3]:
path_diff = "examples/specification/differentiation.schedule.json"
path_v1 = "examples/toy/ACDC.schedule.json"
path_kronecker = "examples/benchmark/kronecker-small.schedule.json"
path_rand_diff = "examples/specification/random-differentiation.schedule.json"

schedule! = Models.load(path_v1, seed="seed123");

In [3]:

# Dryrun ACDC: inspect what segments are generated and their execution paths
sched_acdc = Models.load("examples/toy/ACDC.schedule.json", seed="seed123")

segments = []
sched_acdc(Models.FlatState(); dryrun = (primitive!, x, Δt; path, _...) -> begin
    push!(segments, (
        execution_path = path,
        model_path = primitive!.path,
        from = x.t,
        to = x.t + (isfinite(Δt) ? Δt : 0.0),
        is_instant = !isfinite(Δt) || Δt == 0.0,
    ))
end)

println("Total segments: $(length(segments))\n")
println("$(rpad("execution_path", 30))  $(rpad("model_path", 40))  from -> to")
println("-"^90)
for s in segments
    println("$(rpad(s.execution_path, 30))  $(rpad(s.model_path, 40))  $(s.from) -> $(s.to)$(s.is_instant ? "  [instant]" : "")")
end


Total segments: 17001

execution_path                  model_path                                from -> to
------------------------------------------------------------------------------------------
-1                              -1                                        0.0 -> 0.0  [instant]
-2-1/1+                         -2-1/1.do                                 0.0 -> 1000.0
-2-1/1+                         -2-1/1.do                                 1000.0 -> 2000.0
-2-1/1+                         -2-1/1.do                                 2000.0 -> 3000.0
-2-1/1+                         -2-1/1.do                                 3000.0 -> 4000.0
-2-1/1+                         -2-1/1.do                                 4000.0 -> 5000.0
-2-1/1+                         -2-1/1.do                                 5000.0 -> 6000.0
-2-1/1+                         -2-1/1.do                                 6000.0 -> 7000.0
-2-1/1+                         -2-1/1.do                              

Excessive output truncated after 524302 bytes.

-2-2/11+                        -2-2/11.do                                153000.0 -> 154000.0
-2-2/11+                        -2-2/11.do                                154000.0 -> 155000.0
-2-2/11+                        -2-2/11.do                                155000.0 -> 156000.0
-2-2/11+                        -2-2/11.do                                156000.0 -> 157000.0
-2-2/11+                        -2-2/11.do                                157000.0 -> 158000.0
-2-2/11+                        -2-2/11.do                                158000.0 -> 159000.0
-2-2/11+                        -2-2/11.do                                159000.0 -> 160000.0
-2-2/11+                        -2-2/11.do                                160000.0 -> 161000.0
-2-2/11+                        -2-2/11.do                                161000.0 -> 162000.0
-2-2/11+                        -2-2/11.do                                162000.0 -> 163000.0
-2-2/11+                        -2-2/11.do        

In [ ]:
f! = Scheduling.reify(schedule!, "-2-1+-1+")

In [ ]:
rs = f!.f!.model.model.definition

Reify the schedule. Need to do a dryrun to get all the primitives

In [ ]:
# this only extracts the first network that it finds in the schedule, rather than all of them
function extract_network(schedule::Scheduling.Schedule)
    network = nothing
    
    function dryrun_collector(primitive!, x, Δt; path, _...)
        if !(isfinite(Δt) && Δt > 0.0)
            return
        end
        if network === nothing
            network = Models.NetworkRepresentation.entity(primitive!)
        end
    end
    
    schedule(Models.FlatState(); dryrun=dryrun_collector)
    return network
end

network = extract_network(schedule!)